In [ ]:
#@title فحص وفك ملفات مصدر صراع الأساطير (خلية واحدة)
from google.colab import drive
drive.mount('/content/drive')

import os, json, csv, hashlib, zipfile, pathlib, shutil, time
from collections import Counter

SOURCE_DIR = "/content/drive/MyDrive/Conquer Source ZIPs"
OUT_DIR = "/content/drive/MyDrive/ClashOfLegends_PrivateSourceIntake"
EXPECTED = {
    "Conquer-Mobile-Assets.-mobile-source-files.zip": {
        "size": 250389993,
        "sha256": "5acb29a128f4560bd84a956a628411f12ee713ceb5e2ecb935f65d31ba37d4e2"
    },
    "ConquerOnline.zip": {
        "size": 461117324,
        "sha256": "b1c6958109a07c237631943fc0442d3161563d89e1020b143d95437ba9a6fdb0"
    }
}
os.makedirs(OUT_DIR, exist_ok=True)

def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def safe_extract(zf, destination):
    root=os.path.realpath(destination)
    for info in zf.infolist():
        if info.is_dir(): continue
        target=os.path.realpath(os.path.join(destination, info.filename))
        if not (target == root or target.startswith(root + os.sep)):
            raise RuntimeError("Unsafe ZIP path rejected: " + info.filename)
    zf.extractall(destination)

if not os.path.isdir(SOURCE_DIR):
    raise FileNotFoundError("Source folder not found: " + SOURCE_DIR)

summary={"source_dir":SOURCE_DIR,"output_dir":OUT_DIR,"archives":[],"created_at_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
all_rows=[]
for name, expected in EXPECTED.items():
    src=os.path.join(SOURCE_DIR,name)
    if not os.path.isfile(src):
        summary["archives"].append({"name":name,"status":"MISSING"})
        print("MISSING:",name)
        continue
    size=os.path.getsize(src)
    digest=sha256_file(src)
    integrity=(size==expected["size"] and digest==expected["sha256"])
    record={"name":name,"size_bytes":size,"sha256":digest,"expected_size_bytes":expected["size"],"expected_sha256":expected["sha256"],"source_integrity":"PASS" if integrity else "MISMATCH"}
    print(("PASS" if integrity else "MISMATCH"),name,"|",size,"bytes")
    if not integrity:
        summary["archives"].append(record)
        print("Extraction stopped for this archive because its original checksum/size did not match.")
        continue
    dest=os.path.join(OUT_DIR, pathlib.Path(name).stem)
    os.makedirs(dest,exist_ok=True)
    with zipfile.ZipFile(src,"r") as zf:
        bad=zf.testzip()
        if bad:
            record["zip_integrity"]="FAIL: "+bad
            summary["archives"].append(record)
            print("ZIP CRC ERROR:",bad)
            continue
        infos=[i for i in zf.infolist() if not i.is_dir()]
        record["zip_integrity"]="PASS"
        record["file_count"]=len(infos)
        record["uncompressed_bytes"]=sum(i.file_size for i in infos)
        safe_extract(zf,dest)
    ext_counts=Counter()
    for current,_,files in os.walk(dest):
        for filename in files:
            path=os.path.join(current,filename)
            rel=os.path.relpath(path,OUT_DIR).replace(os.sep,"/")
            ext=pathlib.Path(filename).suffix.lower() or "[no extension]"
            ext_counts[ext]+=1
            all_rows.append({"archive":name,"relative_path":rel,"size_bytes":os.path.getsize(path),"sha256":sha256_file(path),"extension":ext})
    record["extracted_extension_counts"]=dict(sorted(ext_counts.items()))
    record["status"]="EXTRACTED_AND_HASHED"
    summary["archives"].append(record)
    print("EXTRACTED:",record["file_count"],"files |",record["uncompressed_bytes"],"uncompressed bytes")

csv_path=os.path.join(OUT_DIR,"source_asset_inventory.csv")
with open(csv_path,"w",newline="",encoding="utf-8-sig") as f:
    writer=csv.DictWriter(f,fieldnames=["archive","relative_path","size_bytes","sha256","extension"])
    writer.writeheader(); writer.writerows(all_rows)
json_path=os.path.join(OUT_DIR,"source_asset_intake_report.json")
with open(json_path,"w",encoding="utf-8") as f:
    json.dump(summary,f,ensure_ascii=False,indent=2)
print("\nREPORT:",json_path)
print("INVENTORY:",csv_path)
print("TOTAL EXTRACTED FILES:",len(all_rows))
print("IMPORTANT: Source files remain in your private Google Drive. This notebook does NOT publish proprietary client assets to the public GitHub repository.")
